# Afinar el embedding sobre derecho colombiano

Entrena `intfloat/multilingual-e5-large` con pares reales de este proyecto:
consulta coloquial -> fragmento que de verdad la responde, y los candidatos
equivocados que el buscador de hoy trae en su lugar.

**Por qué en Colab y no en local:** el reindexado completo con este modelo
tardó 17,5 h en una T4. Entrenar es más liviano que reindexar, pero en CPU
sigue sin ser viable (medido en este proyecto con modelos mucho más chicos).

**Antes de correr:**
1. `Entorno de ejecución -> Cambiar tipo -> GPU`.
2. Sube `pares_embedding.jsonl` (lo genera
   `finetune/generar_pares_embedding.py`) a la carpeta de Drive de abajo.

**Después:** baja el modelo afinado, ponlo en `finetune/embedding_afinado/`,
reindexa con él y **mide contra `banco_coloquial_prueba.json`**, que no se usó
para entrenar. Si no mejora ahí, no mejoró.

In [ ]:
import torch
assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecución -> Cambiar tipo -> GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install "sentence-transformers>=3.0" accelerate datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# La carpeta NO esta en la raiz de Drive: cuelga de "00_Programas y macros".
# En Colab, MyDrive es "Mi unidad", asi que la ruta lleva esa carpeta en medio.
# Se prueban las rutas conocidas y, si ninguna existe, se busca el archivo: es
# mas barato buscarlo que perder la sesion por una ruta escrita a mano.
import glob, os

NOMBRE = 'pares_embedding.jsonl'
CANDIDATAS = [
    '/content/drive/MyDrive/00_Programas y macros/aliado_libre_eval',
    '/content/drive/MyDrive/aliado_libre_eval',
]
CARPETA = next((c for c in CANDIDATAS if os.path.exists(f'{c}/{NOMBRE}')), None)
if CARPETA is None:
    hallados = glob.glob(f'/content/drive/MyDrive/**/{NOMBRE}', recursive=True)
    assert hallados, (
        f'No encuentro {NOMBRE} en tu Drive. Comprueba que termino de sincronizar '
        '(Drive para escritorio marca los archivos con una nube hasta que suben).'
    )
    CARPETA = os.path.dirname(hallados[0])
ARCHIVO = f'{CARPETA}/{NOMBRE}'
print('usando', ARCHIVO)

print(sum(1 for _ in open(ARCHIVO, encoding='utf-8')), 'pares')
ARCHIVO_PARES = ARCHIVO

## Los prefijos de e5 no son un detalle

Los modelos e5 se entrenaron con `query:` delante de la pregunta y `passage:`
delante del texto. Si se entrena sin ellos y en producción sí se usan (o al
revés), el modelo aprende una cosa y se le pregunta otra — el mismo desajuste
train/inferencia que ya costó un techo de precisión en este proyecto con el
LLM. `index/buscar.py` pone `query:` hoy, así que aquí se hace igual.

In [ ]:
import json
from datasets import Dataset

registros = [json.loads(l) for l in open(ARCHIVO_PARES, encoding='utf-8')]
print(len(registros), 'consultas con sus negativos')

## Antes de entrenar: limpiar los negativos falsos

**Esto no es opcional y es la parte que decide si el afinado ayuda o hace daño.**

Los negativos de este material se sacaron de los primeros resultados de la
búsqueda descartando el fragmento ancla. Esos son precisamente los candidatos con
más probabilidad de ser *también* correctos aunque nadie los haya etiquetado. Una
auditoría con juez sobre 100 consultas de este proyecto encontró que en el **79%**
de los casos alguno de los cinco primeros respondía la pregunta, aunque no fuera
el ancla. En MS-MARCO se midió algo casi idéntico (~70%).

Entrenar con eso le enseña al modelo justo lo contrario de lo que se quiere:
**a alejar del usuario documentos que sí responden.**

La solución es la de RocketQA y NV-Retriever: pasar cada negativo por un
cross-encoder y descartar los que puntúan demasiado alto respecto al positivo.
Aquí se usa el mismo reranker que ya está medido en el proyecto
(`bge-reranker-v2-m3`), con un umbral relativo: se descarta todo negativo que
puntúe por encima del 95% de lo que puntúa el positivo de su propia consulta. El
umbral es relativo y no absoluto porque las puntuaciones del cross-encoder no
son comparables entre consultas.

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np

UMBRAL = 0.95  # un negativo que puntue por encima del 95% del positivo es sospechoso

juez = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512, device='cuda')

limpios, descartados, sin_negativos = [], 0, 0
for d in registros:
    pares = [[d['consulta'], d['positivo']]] + [[d['consulta'], n] for n in d['negativos']]
    puntajes = juez.predict(pares, batch_size=64, show_progress_bar=False)
    corte = puntajes[0] * UMBRAL if puntajes[0] > 0 else puntajes[0] / UMBRAL
    buenos = [n for n, s in zip(d['negativos'], puntajes[1:]) if s < corte]
    descartados += len(d['negativos']) - len(buenos)
    if not buenos:
        sin_negativos += 1
        continue
    limpios.append({**d, 'negativos': buenos})

total = sum(len(d['negativos']) for d in registros)
print(f'negativos originales:  {total}')
print(f'descartados por falsos: {descartados} ({descartados/total*100:.0f}%)')
print(f'consultas que se quedan sin ningun negativo: {sin_negativos}')
print(f'consultas utilizables: {len(limpios)} de {len(registros)}')
registros = limpios

In [ ]:
filas = {"anchor": [], "positive": [], "negative": []}
for d in registros:
    for negativo in d["negativos"]:
        filas["anchor"].append("query: " + d["consulta"])
        filas["positive"].append("passage: " + d["positivo"])
        filas["negative"].append("passage: " + negativo)

datos = Dataset.from_dict(filas)
print(datos)
print(datos[0]["anchor"][:120])

## Entrenamiento

`MultipleNegativesRankingLoss` con tripletas: por cada ejemplo, el resto del
lote también hace de negativo, así que **el tamaño del lote es un
hiperparámetro de calidad**, no solo de velocidad. En una T4 con e5-large a
512 tokens no cabe mucho: se usa lote chico y precisión mixta.

Una sola época. Con ~3.000 tripletas sobre un modelo de 560M, más épocas
memorizan el banco en vez de aprender el salto de vocabulario.

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

modelo = SentenceTransformer('intfloat/multilingual-e5-large')
modelo.max_seq_length = 512

args = SentenceTransformerTrainingArguments(
    output_dir='/content/embedding_afinado',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=20,
    save_strategy='no',
)

SentenceTransformerTrainer(
    model=modelo,
    args=args,
    train_dataset=datos,
    loss=MultipleNegativesRankingLoss(modelo),
).train()

modelo.save_pretrained('/content/embedding_afinado')
print('listo')

## Prueba de humo antes de bajar 2 GB

No mide calidad —para eso está el banco de prueba en local— pero detecta el
fallo tonto: que el modelo guardado no cargue o devuelva vectores degenerados.

In [ ]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('/content/embedding_afinado')
v = m.encode(['query: me echaron del trabajo estando incapacitado',
              'passage: estabilidad laboral reforzada del trabajador en condición de debilidad manifiesta',
              'passage: régimen aduanero de importación temporal'])
import numpy as np
sim = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print('pertinente ', round(sim(v[0], v[1]), 3))
print('impertinente', round(sim(v[0], v[2]), 3))
assert sim(v[0], v[1]) > sim(v[0], v[2]), 'el modelo afinado no distingue: algo salió mal'

In [ ]:
!cd /content && zip -qr embedding_afinado.zip embedding_afinado
!cp /content/embedding_afinado.zip "$CARPETA/"
print('en tu Drive: embedding_afinado.zip')

## Y después, lo único que decide

Reindexar con el modelo afinado y correr el mismo banco de **prueba** (640
consultas que no se usaron aquí). Si el recall@5 no sube ahí, el afinado no
sirvió — por bonita que se vea la curva de pérdida.